In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "8"

In [ ]:
import sys

# 프로젝트 루트를 path에 추가
PROJECT_ROOT = os.path.abspath("")
sys.path.insert(0, PROJECT_ROOT)

print("Project root:", PROJECT_ROOT)


In [ ]:
import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt

from ultralytics import YOLO

from distill.teacher import load_teacher
from distill.hooks import AttentionHook

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

In [ ]:
TEACHER_WEIGHT = "./weights/yolom_kpt.pt"
STUDENT_WEIGHT = "../weights/yolo11n-pose.pt"

teacher = load_teacher(TEACHER_WEIGHT, device)
student = YOLO(STUDENT_WEIGHT).model.to(device)

teacher.eval()
student.eval()


In [ ]:
for i, m in enumerate(student.model):
    print(i, type(m))

In [ ]:
import cv2
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

img_path = "../datasets/images_test/test.jpg"  # 사람 있는 이미지 1장
img = cv2.imread(img_path)
assert img is not None, f"이미지 못 읽음: {img_path}"

img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
img = cv2.resize(img, (640, 640))  # imgsz에 맞추기

# (H, W, C) -> (1, C, H, W), float32, 0~1 정규화
img_tensor = torch.from_numpy(img).permute(2, 0, 1).unsqueeze(0).float().to(device) / 255.0

print(img_tensor.shape, img_tensor.dtype, img_tensor.device)

In [ ]:
t_hook = AttentionHook()
s_hook = AttentionHook()


In [ ]:
teacher.model[-2].register_forward_hook(t_hook)
student.model[-2].register_forward_hook(s_hook)

with torch.no_grad():
    _ = teacher(img_tensor)
    _ = student(img_tensor)

print("Teacher att is None?", t_hook.att is None)
print("Student att is None?", s_hook.att is None)

if t_hook.att is not None:
    print("Teacher att device:", t_hook.att.device)
if s_hook.att is not None:
    print("Student att device:", s_hook.att.device)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def show_att(att, title):
    att = att.detach().cpu().numpy()
    att = (att - att.min()) / (att.max() - att.min() + 1e-6)
    plt.figure(figsize=(4,4))
    plt.imshow(att, cmap="hot")
    plt.title(title)
    plt.axis("off")

show_att(t_hook.att[0], "Teacher Attention")
show_att(s_hook.att[0], "Student Attention")


In [ ]:
def show(att, title):
    a = att.detach().cpu().numpy()
    a = (a - a.min()) / (a.max() - a.min() + 1e-6)
    plt.imshow(a, cmap="hot")
    plt.title(title)
    plt.axis("off")



In [ ]:
show(t_hook.att[0], "Teacher")


In [ ]:
show(s_hook.att[0], "Student")